# Hispasonic ETL — Unified Dataset

**Source:** 12 CSV scrapes from Hispasonic (2022–2024)  
**Goal:** Load, inspect, clean and consolidate into a single homogeneous dataset  
**Output:** `../data/processed/hispasonic_unified.csv`

---

## Pipeline
1. Load all CSVs
2. Inspect columns and detect inconsistencies
3. Normalize to common schema (16 core columns)
4. Basic cleaning (types, nulls, duplicates)
5. Save unified dataset

## 1. Imports and config

In [ ]:
import pandas as pd
import os
import glob

RAW_DIR = '../data/raw/'
PROCESSED_DIR = '../data/processed/'

# Core columns — common denominator across all files
CORE_COLUMNS = [
    'urgent', 'buy', 'change', 'sell', 'price', 'gift',
    'search', 'repair', 'parts', 'synt_brand', 'description',
    'city', 'published', 'expire', 'date_scrapped', 'seen'
]

print('pandas version:', pd.__version__)
print('Raw dir:', os.path.abspath(RAW_DIR))

## 2. Load and inspect each CSV

In [ ]:
csv_files = sorted(glob.glob(os.path.join(RAW_DIR, '*.csv')))
print(f'{len(csv_files)} files found:\n')

for f in csv_files:
    df = pd.read_csv(f, nrows=0)
    cols = list(df.columns)
    extra = [c for c in cols if c not in CORE_COLUMNS]
    missing = [c for c in CORE_COLUMNS if c not in cols]
    print(f'{os.path.basename(f)}')
    print(f'  total cols: {len(cols)}')
    print(f'  extra cols: {extra}')
    print(f'  missing core cols: {missing}')
    print()

## 3. Load all CSVs and normalize to core schema

In [ ]:
frames = []

for f in csv_files:
    df = pd.read_csv(f)
    
    # Drop unnamed index column if present
    unnamed = [c for c in df.columns if c.startswith('Unnamed')]
    df = df.drop(columns=unnamed)
    
    # Keep only core columns that exist in this file
    available = [c for c in CORE_COLUMNS if c in df.columns]
    df = df[available]
    
    # Tag source file for traceability
    df['source_file'] = os.path.basename(f)
    
    frames.append(df)
    print(f'{os.path.basename(f)}: {len(df)} rows, {len(df.columns)} cols')

print(f'\nTotal frames loaded: {len(frames)}')

## 4. Concatenate into unified dataset

In [ ]:
df_all = pd.concat(frames, ignore_index=True)

print(f'Unified shape: {df_all.shape}')
print(f'Columns: {list(df_all.columns)}')
df_all.head()

## 5. Inspect nulls and data types

In [ ]:
print('=== Null counts ===')
print(df_all.isnull().sum())
print()
print('=== Data types ===')
print(df_all.dtypes)
print()
print('=== Basic stats ===')
df_all.describe(include='all')

## 6. Fix data types

In [ ]:
# Date columns
for col in ['published', 'expire', 'date_scrapped']:
    df_all[col] = pd.to_datetime(df_all[col], infer_datetime_format=True, errors='coerce')

# Numeric columns
df_all['price'] = pd.to_numeric(df_all['price'], errors='coerce')
df_all['seen'] = pd.to_numeric(df_all['seen'], errors='coerce').astype('Int64')

# Boolean-like columns (should be 0/1)
bool_cols = ['urgent', 'buy', 'change', 'sell', 'gift', 'search', 'repair', 'parts']
for col in bool_cols:
    df_all[col] = pd.to_numeric(df_all[col], errors='coerce').astype('Int64')

print('Types after conversion:')
print(df_all.dtypes)

## 7. Check for duplicate rows

In [ ]:
subset_cols = ['description', 'price', 'published', 'city']
n_dupes = df_all.duplicated(subset=subset_cols).sum()
print(f'Duplicate rows (by {subset_cols}): {n_dupes}')

# Show duplicates if any
if n_dupes > 0:
    df_all[df_all.duplicated(subset=subset_cols, keep=False)].sort_values('description').head(20)

## 8. Save unified dataset

In [ ]:
os.makedirs(PROCESSED_DIR, exist_ok=True)
output_path = os.path.join(PROCESSED_DIR, 'hispasonic_unified.csv')

df_all.to_csv(output_path, index=False)

print(f'Saved: {output_path}')
print(f'Shape: {df_all.shape}')
print(f'Date range: {df_all["date_scrapped"].min()} → {df_all["date_scrapped"].max()}')